# BloomFace · Generación floral guiada

Genera una acuarela nueva al recibir una expresión confirmada. Stable Diffusion 1.5 usa
ControlNet Scribble para guiar la forma y LCM-LoRA para generar en **4 pasos de inferencia**.
El filtro permanece activo. Se generan imágenes en memoria a partir de guías dibujadas por el equipo.

Abre el cuaderno en **Google Colab con GPU**, ejecuta las celdas en orden y repite solo la última
para probar otro gesto. La primera carga descarga los pesos públicos. Si preparas el cuaderno en una
sesión en la que ya importaste estas librerías, reiníciala tras instalar las dependencias.

[Stable Diffusion 1.5](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5)
· [ControlNet Scribble](https://huggingface.co/lllyasviel/control_v11p_sd15_scribble)
· [LCM-LoRA](https://huggingface.co/latent-consistency/lcm-lora-sdv1-5)

## 1. Preparar Colab

In [ ]:
%pip install -q "diffusers==0.42.1" "peft==0.21.2" transformers accelerate safetensors
%pip uninstall -y torchao

## 2. Cargar el modelo y activar LCM

In [ ]:
from random import SystemRandom
from time import perf_counter

import torch
from diffusers import (
    ControlNetModel,
    LCMScheduler,
    StableDiffusionControlNetPipeline,
    StableDiffusionPipeline,
)
from IPython.display import display
from PIL import Image, ImageDraw

if not torch.cuda.is_available():
    raise RuntimeError("Activa una GPU en Colab y vuelve a ejecutar esta celda.")

modelo_base = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_scribble",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
editor_guiado = StableDiffusionControlNetPipeline.from_pipe(
    modelo_base, controlnet=controlnet
).to("cuda")
editor_guiado.load_lora_weights(
    "latent-consistency/lcm-lora-sdv1-5",
    weight_name="pytorch_lora_weights.safetensors",
    adapter_name="lcm_rapido",
)
editor_guiado.set_adapters("lcm_rapido", adapter_weights=1.0)
editor_guiado.enable_lora()
editor_guiado.scheduler = LCMScheduler.from_config(editor_guiado.scheduler.config)

flor_visible = None

## 3. Configurar las cuatro expresiones, los prompts y las guías

In [ ]:
contornos = {
    "Cerrada": [
        (256, 270), (225, 248), (211, 214), (207, 176),
        (219, 137), (238, 112), (256, 100), (274, 112),
        (293, 137), (305, 176), (301, 214), (287, 248),
        (256, 270),
    ],
    "Punto medio": [
        (256, 270), (223, 249), (201, 219), (186, 181),
        (183, 140), (187, 112), (214, 129), (236, 156),
        (256, 122), (276, 156), (298, 129), (325, 112),
        (329, 140), (326, 181), (311, 219), (289, 249),
        (256, 270),
    ],
}

contornos_nuevos = {
    "Abierta": [
        (256, 270), (213, 250), (176, 216), (151, 173),
        (143, 130), (173, 145), (207, 159), (232, 132),
        (256, 149), (280, 132), (305, 159), (339, 145),
        (369, 130), (361, 173), (336, 216), (299, 250),
        (256, 270),
    ],
    "Más abierta": [
        (256, 270), (215, 251), (178, 228), (135, 208),
        (89, 176), (66, 145), (111, 153), (156, 169),
        (203, 165), (230, 132), (256, 146), (282, 132),
        (309, 165), (356, 169), (401, 153), (446, 145),
        (423, 176), (377, 208), (334, 228), (297, 251),
        (256, 270),
    ],
}

formas = {
    "Cerrada": "tight closed tulip bud, overlapping upright petals, hidden center",
    "Punto medio": "partly open tulip, narrow cup, separated petal tips at the top",
}

formas_nuevas = {
    "Abierta": "open tulip forming a broad cup, upright curved petals, visible center",
    "Más abierta": (
        "wide flared tulip, outer petals extending sideways almost horizontally, "
        "shallow bowl and clearly visible center"
    ),
}

estilo = (
    "botanical watercolor illustration, single pink tulip from the side, "
    "whole green stem, soft translucent washes, plain ivory paper"
)


etapa_por_expresion = {
    "ceno_fruncido": "Cerrada",
    "reposo": "Punto medio",
    "sonrisa": "Abierta",
    "boca_abierta": "Más abierta",
}

descripciones = {**formas, **formas_nuevas}
guias = {}
for etapa, contorno in {**contornos, **contornos_nuevos}.items():
    guia = Image.new("RGB", (512, 512), "black")
    dibujo = ImageDraw.Draw(guia)
    dibujo.line(contorno, fill="white", width=5, joint="curve")
    dibujo.line([(256, 270), (254, 335), (256, 410)], fill="white", width=5)
    guias[etapa] = guia

## 4. Generar una acuarela por expresión confirmada

In [ ]:
def generar_flor(expresion, flor_actual=None, *, pasos=4, semillas=None):
    """Devuelve la imagen que debe quedar visible y el registro de intentos."""
    etapa = etapa_por_expresion[expresion]
    if semillas is None:
        semillas = SystemRandom().sample(range(2**32), k=2)
    elif len(semillas) != 2 or semillas[0] == semillas[1]:
        raise ValueError("Indica dos semillas distintas para la prueba.")

    intentos = []
    for semilla in semillas:
        generador = torch.Generator(device="cuda").manual_seed(semilla)
        torch.cuda.synchronize()
        inicio = perf_counter()

        salida = editor_guiado(
            prompt=f"{descripciones[etapa]}, {estilo}",
            image=guias[etapa],
            controlnet_conditioning_scale=0.9,
            num_inference_steps=pasos,
            guidance_scale=1.0,
            generator=generador,
        )

        torch.cuda.synchronize()
        segundos = perf_counter() - inicio
        imagen = salida.images[0].convert("RGB")
        if salida.nsfw_content_detected and salida.nsfw_content_detected[0]:
            estado = "Marcada por el filtro"
        elif imagen.getbbox() is None:
            estado = "Salida negra"
        else:
            estado = "Imagen aceptada"

        intentos.append((semilla, segundos, estado))
        if estado == "Imagen aceptada":
            return imagen, intentos

    return flor_actual, intentos

## 5. Probar una expresión

Cambia la expresión y vuelve a ejecutar esta celda. La generación usa semillas nuevas de forma
predeterminada. Para repetir el diagnóstico anterior de la flor cerrada, usa
**ceno_fruncido** y fija las semillas en **(42, 83)**. Ajusta **pasos** a **8** si quieres probar
esa configuración. La versión validada para interacción usa **4**.

In [ ]:
expresion_confirmada = "ceno_fruncido"
pasos = 4
semillas_prueba = None

flor_anterior = flor_visible
inicio_total = perf_counter()
flor_visible, intentos = generar_flor(
    expresion_confirmada,
    flor_actual=flor_anterior,
    pasos=pasos,
    semillas=semillas_prueba,
)
segundos_totales = perf_counter() - inicio_total

print(f"{expresion_confirmada} → {etapa_por_expresion[expresion_confirmada]} · {pasos} pasos")
for semilla, segundos, estado in intentos:
    print(f"Semilla {semilla} · {segundos:.2f} s · {estado}")
print(f"Tiempo total: {segundos_totales:.2f} s")

if flor_visible is None:
    print("No se obtuvo una imagen utilizable.")
else:
    print("Se conserva la flor anterior." if flor_visible is flor_anterior else "Nueva flor visible.")
    display(flor_visible)